# Induction Eval — Results Analysis
Loads the YAML result files produced by `induction_eval.ipynb` and plots
accuracy for each (model × context-type) combination in a single figure.

**HISTORICAL FIGURE — pinned to the archived pilot data.** This notebook
reads the archived single-run pilot's flat `result2/{tag}_{info}.yaml`
files from the ORIGINAL model trio (devstral-small-24b / gemma-3-27b-it /
qwen3-30b-a3b). The pilot never produced the three `noise_intens`
conditions, so those print as "missing" below — that is expected, not a
bug. Do **not** point this notebook at `results/`: the current experiment
writes a different data shape (per-replicate `results/{tag}_{info}/
rep_{seed}.yaml` directories) for a different model trio; see the loading
cell's own banner comment for the full rationale.


In [ ]:
from pathlib import Path

import smolbench
from smolbench.induction.figures import load_condition_accuracies

# This notebook now anchors its paths off the installed `smolbench` package
# rather than `sys.path.insert(0, str(Path("..").resolve()))` + a cwd-relative
# `Path("result2")`: the old hack only worked when the notebook's kernel was
# started with `notebooks/chromatic/` as its cwd, and silently broke (wrong
# `result2/`, or an unimportable `smolbench.evals`) whenever it wasn't.
# `smolbench.__file__` is stable regardless of the kernel's cwd.
NB = Path(smolbench.__file__).resolve().parents[1] / "notebooks" / "chromatic"

# HISTORICAL FIGURE -- pinned to the archived single-run pilot (the flat
# result2/{tag}_{info}.yaml files; the pilot never produced the three
# noise_intens conditions, which therefore print as missing) produced by the
# ORIGINAL trio (devstral-small-24b / gemma-3-27b-it / qwen3-30b-a3b) BEFORE
# the R=30 replicate redesign. The current experiment (olmo-3.1/granite
# trio) writes per-replicate results/{tag}_{info}/rep_{seed}.yaml dirs,
# which this notebook deliberately does NOT read -- aggregate those with
# summarize() in induction_eval.ipynb. Only regenerate the PNG against the
# archived flat files; pointing it at the replicate dirs would attach the
# legacy model labels below to data they never produced.
RESULTS_DIR = NB / "result2"

# (model_key, condition_key) -> result file
FILES = {
    ("decode", "intens"):        "decode_intens.yaml",
    ("decode", "extens"):        "decode_extens.yaml",
    ("decode", "noise_intens"):  "decode_noise_intens.yaml",
    ("cot",    "intens"):        "cot_intens.yaml",
    ("cot",    "extens"):        "cot_extens.yaml",
    ("cot",    "noise_intens"):  "cot_noise_intens.yaml",
    ("moe",    "intens"):        "moe_intens.yaml",
    ("moe",    "extens"):        "moe_extens.yaml",
    ("moe",    "noise_intens"):  "moe_noise_intens.yaml",
}

# Loading loop (missing-file printing, accuracy scoring) now lives in
# smolbench.induction.figures.load_condition_accuracies, shared with any
# future analysis notebook that needs the same (model, condition) -> accuracy
# table shape.
data = load_condition_accuracies(RESULTS_DIR, FILES)
data


In [ ]:
import matplotlib.pyplot as plt

from smolbench.induction.figures import plot_archetype_accuracy

MODELS = [
    ("decode", "Decoder-only\n(devstral-small-24b)"),
    ("cot",    "CoT\n(gemma-3-27b-it)"),
    ("moe",    "MoE\n(qwen3-30b-a3b)"),
]

CONDITIONS = [
    ("intens",       "Intensional",           "#4C72B0"),
    ("extens",       "Extensional",           "#DD8452"),
    ("noise_intens", "Intensional + Noise",   "#55A868"),
]

# Grouped-bar figure builder now lives in smolbench.induction.figures; all
# defaults (bar width, figure size, y-limits, 50% chance line) match this
# cell's original hardcoded values, so this call reproduces the pinned
# figure exactly (see tests/test_induction_figures.py).
fig, ax = plot_archetype_accuracy(
    data, MODELS, CONDITIONS,
    title="Induction Eval: Accuracy by Model and Context Type NIAH",
    out_path=NB / "induction_eval_results.png",
)
plt.show()
